# ガゼッティア（辞書）による固有表現認識とサブワード・BIO ラベル整合

書籍 第3章「系列ラベリング問題の解き方」（3-1-1 ルールベース、3-1-3 深層学習ベース）に対応する演習ノートブックです。
CPU のみ・数秒で実行でき、実務で頻出する 2 つの設計ポイントを体験します。

1. **ガゼッティア（固有表現辞書）と最長一致・多義語誤マッチ対策（ハイブリッド判定）**（3-1-1、p.85）
2. **BERT 等のサブワード分割と BIO タグの整合処理・不正遷移補正**（3-1-3、p.87〜88）

In [1]:
!pip install -q sentencepiece

In [2]:
import tempfile
from pathlib import Path

import pandas as pd
import sentencepiece as spm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

---

## 1. ガゼッティア（最長一致）と多義語による誤認識（3-1-1、p.85）

地名辞典・人名・組織名リスト（**ガゼッティア**）を用いた辞書マッチングでは、短い単語（「東京」「日本」）が長い固有名詞（「東京大学」「日本銀行」）の一部を誤って抜かないよう**最長一致**を使うのが基本です。
しかしそれでも、「青森（地名 vs りんご品種）」「千葉（地名 vs 苗字）」「アップル（IT企業 vs 果物）」のような**多義語**は文字列一致だけでは誤認識してしまいます。

In [3]:
gazetteer = {
    "東京": "LOC",
    "東京大学": "ORG",
    "日本": "LOC",
    "日本橋": "LOC",
    "日本銀行": "ORG",
    "青森": "LOC",
    "千葉": "LOC",
    "アップル": "ORG",
    "鈴木": "PER",
}

def extract_longest_matches(text, dic):
    results = []
    i = 0
    # 長い見出し語から優先して照合する
    keys = sorted(dic.keys(), key=len, reverse=True)
    while i < len(text):
        matched = None
        for k in keys:
            if text.startswith(k, i):
                matched = k
                break
        if matched:
            results.append((i, i + len(matched), matched, dic[matched]))
            i += len(matched)
        else:
            i += 1
    return results

sentences = [
    "東京大学の鈴木教授が日本銀行を訪れた。",
    "青森は甘みと酸味のバランスが良いりんごです。",
    "千葉さんがスーパーでアップルパイを買った。",
]

for s in sentences:
    print("文:", s)
    print("  最長一致結果:", [(m, tag) for _, _, m, tag in extract_longest_matches(s, gazetteer)])

文: 東京大学の鈴木教授が日本銀行を訪れた。
  最長一致結果: [('東京大学', 'ORG'), ('鈴木', 'PER'), ('日本銀行', 'ORG')]
文: 青森は甘みと酸味のバランスが良いりんごです。
  最長一致結果: [('青森', 'LOC')]
文: 千葉さんがスーパーでアップルパイを買った。
  最長一致結果: [('千葉', 'LOC'), ('アップル', 'ORG')]


### ハイブリッドアプローチ：辞書候補 ＋ 文脈による 2 値分類（3-1-1、p.85）

書籍 p.85 では、ノイジーなテキストに対して**「まずガゼッティアで候補文字列を取り出し、その前後の文脈から実際にその固有表現か否かを機械学習分類器で判定する」**ハイブリッド手法が紹介されています。簡易な文脈分類器で先ほどの誤マッチを抑制できるか試しましょう。

In [4]:
# (文, 注目スパンが辞書タグ通りの固有表現か: 1=はい, 0=いいえ)
train_cands = [
    ("青森県でリンゴを収穫した。", "青森", 1),
    ("夏休みに青森へ旅行した。", "青森", 1),
    ("青森出身の友人と会った。", "青森", 1),
    ("この品種の青森はとても甘い。", "青森", 0),
    ("青森とふじを食べ比べた。", "青森", 0),
    ("千葉県でイベントが開催された。", "千葉", 1),
    ("千葉駅から電車に乗った。", "千葉", 1),
    ("来週は千葉に出張する。", "千葉", 1),
    ("千葉さんが資料を作成した。", "千葉", 0),
    ("千葉部長に確認をお願いした。", "千葉", 0),
    ("アップルが新型端末を発表した。", "アップル", 1),
    ("アップル社の株価が上昇した。", "アップル", 1),
    ("アップルに転職した。", "アップル", 1),
    ("甘いアップルパイを焼いた。", "アップル", 0),
    ("アップルティーを注文した。", "アップル", 0),
]

def span_context_feature(text, start, end):
    left = text[max(0, start - 4):start]
    right = text[end:min(len(text), end + 5)]
    surf = text[start:end]
    return f"L:{left} M:{surf} R:{right}"

train_feats = []
train_y = []
for text, target, label in train_cands:
    idx = text.index(target)
    train_feats.append(span_context_feature(text, idx, idx + len(target)))
    train_y.append(label)

vec = TfidfVectorizer(analyzer="char", ngram_range=(1, 3))
X_tr = vec.fit_transform(train_feats)
clf = LogisticRegression(C=3.0, max_iter=1000).fit(X_tr, train_y)

test_sentences = [
    "千葉に出張して青森県産のお土産を買った。",
    "千葉さんが会議室でアップルパイを食べた。",
    "アップルが開発した新しいOSが公開された。",
]

for s in test_sentences:
    raw = extract_longest_matches(s, gazetteer)
    filtered = []
    for st, ed, surf, tag in raw:
        feat = span_context_feature(s, st, ed)
        prob = clf.predict_proba(vec.transform([feat]))[0, 1]
        if prob >= 0.5:
            filtered.append((surf, tag, round(float(prob), 2)))
    print("文:", s)
    print("  辞書のみ    :", [(surf, tag) for _, _, surf, tag in raw])
    print("  ハイブリッド:", filtered)

文: 千葉に出張して青森県産のお土産を買った。
  辞書のみ    : [('千葉', 'LOC'), ('青森', 'LOC')]
  ハイブリッド: [('千葉', 'LOC', 0.68), ('青森', 'LOC', 0.69)]
文: 千葉さんが会議室でアップルパイを食べた。
  辞書のみ    : [('千葉', 'LOC'), ('アップル', 'ORG')]
  ハイブリッド: []
文: アップルが開発した新しいOSが公開された。
  辞書のみ    : [('アップル', 'ORG')]
  ハイブリッド: [('アップル', 'ORG', 0.69)]


「千葉**に**出張」「アップル**が**開発」は採用され、「千葉**さん**」「アップル**パイ**」のような別義表現を文脈から棄却できていることがわかります。

---

## 2. サブワード分割と BIO タグ整合・不正遷移の補正（3-1-3、p.88）

BERT などの事前学習モデルで固有表現認識を解く際は、単語単位のアノテーション（BIO タグ）とモデル内部の**サブワード分割**を整合させる前処理・後処理が不可欠です。

1. **アラインメント（訓練・入力時）**: 1 単語が複数のサブワードに分割されたとき、**先頭サブワードだけに元の BIO タグを割り当て、2 番目以降のサブワードには損失計算から除外する `-100` を割り当てる**
2. **不正遷移の補正（推論・後処理時、3-1-3 p.88「I ラベルは単独では出現しない」）**: `B-LOC` がないのに `O` の直後に `I-LOC` が出るような不正な BIO 遷移を `B-LOC` に補正する

In [5]:
# 小規模コーパスで SentencePiece（サブワード分割器）を学習
corpus_lines = [
    "東京大学の鈴木先生が学会で講演した。",
    "日本銀行の本店は東京都中央区日本橋にある。",
    "2月29日にロンドンから空港を経由して帰国する。",
    "ロンドン市内のホテルから空港へ向かい帰国する。",
    "ロンドンから空港まではバスで移動する。",
    "ロンドンでは国際会議に出席して翌日に帰国する。",
    "空港の国際線ロビーで友人と合流した。",
    "羽田空港から成田空港までバスで移動した。",
    "今日は2月15日で、来週にロンドンへ出発する。",
    "ホテルから空港まで電車で移動する。",
    "コーヒーとチーズケーキをカフェで注文する。",
    "スポーツとニュースのテレビ番組を毎日見る。",
]
tmpdir = Path(tempfile.mkdtemp())
corpus_file = tmpdir / "corpus.txt"
corpus_file.write_text("\n".join(corpus_lines), encoding="utf-8")
spm.SentencePieceTrainer.Train(
    input=str(corpus_file),
    model_prefix=str(tmpdir / "sp"),
    vocab_size=115,
    model_type="unigram",
    character_coverage=1.0,
    add_dummy_prefix=False,
    minloglevel=2,
)
sp = spm.SentencePieceProcessor(model_file=str(tmpdir / "sp.model"))

# 単語単位の文と BIO ラベル（3-1 図 3.2 の例文）
words = ["2", "月", "29", "日", "に", "ロンドン", "ヒースロー", "空港", "から", "帰国", "する", "。"]
word_tags = ["B-TIME", "I-TIME", "I-TIME", "I-TIME", "O", "B-LOC", "B-LOC", "I-LOC", "O", "O", "O", "O"]

subwords = []
aligned_labels = []
word_ids = []

for w_idx, (w, tag) in enumerate(zip(words, word_tags)):
    pieces = sp.encode_as_pieces(w)
    for p_idx, p in enumerate(pieces):
        subwords.append(p)
        word_ids.append(w_idx)
        # 先頭サブワードだけタグを残し、後続サブワードは損失計算を無視 (-100) にする
        aligned_labels.append(tag if p_idx == 0 else "-100")

display(pd.DataFrame({
    "word_id": word_ids,
    "元単語": [words[i] for i in word_ids],
    "サブワード": subwords,
    "整合後ラベル": aligned_labels,
}))

,word_id,元単語,サブワード,整合後ラベル
0,0,2,2,B-TIME
1,1,月,月,I-TIME
2,2,29,2,I-TIME
3,2,29,9,-100
4,3,日,日,I-TIME
5,4,に,に,O
6,5,ロンドン,ロンドン,B-LOC
7,6,ヒースロー,ヒ,B-LOC
8,6,ヒースロー,ー,-100
9,6,ヒースロー,ス,-100


In [6]:
def repair_bio_sequence(tags):
    """書籍 3-1-3 (p.88): I ラベルは単独では出現しないため、先頭や O の直後の I-XXX を B-XXX へ補正する"""
    repaired = []
    prev_type = None
    for tag in tags:
        if tag.startswith("I-"):
            ent_type = tag[2:]
            if prev_type != ent_type:
                repaired.append(f"B-{ent_type}")
                prev_type = ent_type
                continue
        if tag.startswith("B-"):
            prev_type = tag[2:]
        elif tag == "O":
            prev_type = None
        else:
            prev_type = tag[2:]
        repaired.append(tag)
    return repaired

# 例えばヒースロー(B-LOC) が O に誤予測され、空港(I-LOC) だけ残ったケース
noisy_pred = ["O", "O", "O", "O", "O", "B-LOC", "O", "I-LOC", "O", "O", "O", "O"]
fixed_pred = repair_bio_sequence(noisy_pred)

print("補正前の単語ラベル:", list(zip(words[5:9], noisy_pred[5:9])))
print("補正後の単語ラベル:", list(zip(words[5:9], fixed_pred[5:9])))

補正前の単語ラベル: [('ロンドン', 'B-LOC'), ('ヒースロー', 'O'), ('空港', 'I-LOC'), ('から', 'O')]
補正後の単語ラベル: [('ロンドン', 'B-LOC'), ('ヒースロー', 'O'), ('空港', 'B-LOC'), ('から', 'O')]
